In [11]:
from functions import *

In [12]:
a_vec = [763, 679, 397, 61, 697, 373, 
         289, 257, 625, 41, 193, 449]
b_vec = [435, 69, 330, 18, 612, 246, 
         496, 640, 200, 524, 672, 672] 

In [13]:
G = generate_g(a_vec)
all_indices = [i for i in range(l_h**2)]
gb_indices = [3, 8]
ga_indices = [i for i in all_indices if i not in gb_indices]
Ga = G.extract(ga_indices, list(range(G.cols)))
Gb = G.extract(gb_indices, list(range(G.cols)))
basis_a = solve_modular_kernel(Ga, P)
V = Matrix.hstack(*basis_a)

In [14]:
# 1. b_vec を行列形式に変換
b_mat = Matrix(b_vec)

# 2. V * x = b を有理数体（Q）上で解く
# V は main.ipynb で定義された Matrix.hstack(*basis_a)
try:
    # V が正則であれば x = V^-1 * b が求まる
    x_rational = V.solve(b_mat)

    # 3. 有理数の解 a/b を整数 a * inv(b, P) (mod P) に変換する関数
    def to_mod_p(val, p):
        num, den = val.as_numer_denom()
        # Python 3.8+ の pow(den, -1, p) はモジュラ逆数を計算する
        return (int(num) * pow(int(den), -1, p)) % p

    # 各要素に適用
    coefficients = x_rational.applyfunc(lambda v: to_mod_p(v, P))

    print("線形結合の係数ベクトル x:")
    display(coefficients)
    
    # 検算: V * x % P が b_vec と一致するか確認
    check_val = (V * coefficients).applyfunc(lambda x: x % P)
    if check_val == b_mat.applyfunc(lambda x: x % P):
        print("検算成功: 一致しました。")
    else:
        print("警告: 検算に失敗しました。")

except Exception as e:
    print(f"解を求めることができませんでした: {e}")

線形結合の係数ベクトル x:


Matrix([
[  3],
[709],
[689],
[746],
[762],
[732],
[ 68],
[ 84],
[565],
[557],
[744],
[346]])

検算成功: 一致しました。


In [15]:
cycles = generate_cycles(6)
h_x, h_z = generate_h_xz()
constraints = generate_constraints(cycles, a_vec, h_x, h_z)

In [16]:
# 全ての禁止ベクトル（法ベクトル）を個別にリスト化する
unique_forbidden_vectors = []
seen_vectors = set()

# 1. 条件B (潜在部の非可換性) からの制約 r_i
for i in range(Gb.rows):
    c_prime = (Gb.row(i) * V).applyfunc(lambda x: x % P)
    c_tuple = tuple(c_prime)
    if c_tuple not in seen_vectors:
        unique_forbidden_vectors.append(c_prime.T) # 列ベクトルとして保存
        seen_vectors.add(c_tuple)

# 2. 条件C (短いサイクルの回避) からの制約 c_prime
for c in constraints:
    c_prime = (Matrix([c]) * V).applyfunc(lambda x: x % P)
    c_tuple = tuple(c_prime)
    if c_tuple not in seen_vectors:
        unique_forbidden_vectors.append(c_prime.T)
        seen_vectors.add(c_tuple)

print(f"個別に回避すべき禁止制約（超平面）の数: {len(unique_forbidden_vectors)}")

個別に回避すべき禁止制約（超平面）の数: 382


In [17]:
def is_in_general_solution(x_vec, forbidden_vectors, p):
    """
    x_vec がすべての禁止超平面 r_i^T * x = 0 (mod p) を避けているか判定する。
    """
    # ベクトル形式を整える
    x_mat = Matrix(x_vec)
    
    for r in forbidden_vectors:
        # 内積が 0 (mod P) になったらその禁止領域に含まれている
        if (r.T * x_mat)[0] % p == 0:
            return False
    return True

# すでに見つけている特殊解 coefficients (x0) の妥当性を再確認
if is_in_general_solution(coefficients, unique_forbidden_vectors, P):
    print("特殊解 x0 は一般解の条件をすべて満たしています。")

In [ ]:
# --- main.ipynb 続き: 一般解集合の構築 ---

def construct_general_solution_space(x0, forbidden_vectors, p, sample_limit=100):
    """
    特殊解 x0 を核として、CRT分解を利用し一般解の集合を構築する
    """
    da = x0.rows
    x0_3 = x0.applyfunc(lambda x: x % 3)
    x0_256 = x0.applyfunc(lambda x: x % 256)
    
    valid_x3_components = []
    
    # 特殊解の周辺 (ハミング距離が小さい範囲) から探索を開始
    # これは全探索ではなく、特殊解の「安全性」を継承する領域の抽出
    for dist in range(da + 1):
        if len(valid_x3_components) >= sample_limit:
            break
            
        # 距離 dist の変化を与える
        for indices in itertools.combinations(range(da), dist):
            for values in itertools.product([-1, 1], repeat=dist):
                dx = Matrix.zeros(da, 1)
                for idx, val in zip(indices, values):
                    dx[idx] = val
                
                x3_cand = (x0_3 + dx).applyfunc(lambda x: x % 3)
                
                # 法 768 で合成してチェック
                x_full = crt_combine_vectors(x3_cand, x0_256) # 前述のCRT合成関数
                
                if is_in_general_solution(x_full, forbidden_vectors, p):
                    valid_x3_components.append(x3_cand)
                    if len(valid_x3_components) >= sample_limit:
                        break
    
    return valid_x3_components

# 一般解を構成する mod 3 成分のリストを取得
S3_components = construct_general_solution_space(coefficients, unique_forbidden_vectors, P)

print(f"条件A, B, Cをすべて満たす一般解の成分集合 S3 (サイズ: {len(S3_components)}) を特定しました。")

In [ ]:
# --- main.ipynb 続き: 解 x の合成と b_vec への復元 ---

# 1. 解ベクトル x (mod 768) のリストを合成
# S3_components の各要素 x3 と、特殊解から抽出した x0_256 を CRT で結合する
x0_256 = coefficients.applyfunc(lambda x: x % 256)
final_x_solutions = [crt_combine_vectors(x3, x0_256) for x3 in S3_components]

def reconstruct_b_vectors(x_solutions, V_matrix, p):
    """
    da 次元の解 x から L 次元の係数ベクトル b を復元する
    b = V * x (mod p)
    """
    b_vectors = []
    for x_sol in x_solutions:
        # 基底行列 V との積を計算し mod P を適用
        b_mat = (V_matrix * x_sol).applyfunc(lambda val: val % p)
        # Matrix 型から Python のリスト形式に変換
        b_list = [int(val) for val in b_mat]
        b_vectors.append(b_list)
    return b_vectors

# 2. L 次元の係数ベクトル b_vec のリストを算出
final_b_list = reconstruct_b_vectors(final_x_solutions, V, P)

print(f"一般解の成分集合 S3 から {len(final_b_list)} 個の b_vec を復元しました。")

一般解の成分集合 S3 から 0 個の b_vec を復元しました。


In [ ]:
# 生成された一般解の一つを b_vec に戻して確認
if final_b_list:
    for b_vec in final_b_list:
        
        print("b_vec:", list(b_vec))
        print("check:", check(a_vec, b_vec), '\n')

